In [1]:
import wrds
import nasdaqdatalink
import pandas as pd
import numpy as np
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")
db = wrds.Connection(wrds_username='ethanyxhuang')
nasdaqdatalink.ApiConfig.api_key = "Mwemn6Joi9zKwmsaRb7N"

d:\Anaconda3\envs\finm\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Loading library list...
Done


In [2]:
mag7 = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA", 'FB']
prices = nasdaqdatalink.get_table(
    "NDAQ/USEDHADJ",
    symbol=mag7,
    date={"gte": "2010-01-01", "lte": "2025-12-31"},
    qopts={"columns": ["symbol", "date", "close_adj", "volume"]},
    paginate=True
)

prices["date"] = pd.to_datetime(prices["date"])
prices = prices.sort_values(["symbol", "date"]).reset_index(drop=True)
meta1 = prices[(prices["symbol"] == "FB") & (prices["date"] <= "2022-06-08")]
meta2 = prices[(prices["symbol"] == "META") & (prices["date"] >= "2022-06-09")]
meta = pd.concat([meta1, meta2], ignore_index=True).sort_values("date").reset_index(drop=True)
meta["symbol"] = "META"
prices = prices[(prices["symbol"] != "FB") & (prices["symbol"] != "META")]
prices = pd.concat([prices, meta], ignore_index=True).sort_values(["symbol", "date"]).reset_index(drop=True)
prices

,symbol,date,close_adj,volume
0,AAPL,2014-01-02,17.141,8387258.0
1,AAPL,2014-01-03,16.765,14043410.0
2,AAPL,2014-01-06,16.856,14765593.0
3,AAPL,2014-01-07,16.736,11347538.0
4,AAPL,2014-01-08,16.842,9240955.0
...,...,...,...,...
21065,TSLA,2025-12-24,485.400,41285429.0
21066,TSLA,2025-12-26,475.190,58780659.0
21067,TSLA,2025-12-29,459.640,66263033.0
21068,TSLA,2025-12-30,454.430,59238464.0


In [3]:
query = """
SELECT permno, ticker, namedt, nameendt, comnam
FROM crsp.msenames
WHERE ticker IN ('AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA', 'FB')
"""

mag7_permno = db.raw_sql(query)
mag7_permno.drop_duplicates(subset=['permno', 'ticker'], inplace=True)
meta = mag7_permno[mag7_permno['comnam'].str.contains('FACEBOOK')]
mag7_permno = mag7_permno[(mag7_permno['ticker'] != 'FB') & (mag7_permno['ticker'] != 'META')]
mag7_permno = pd.concat([mag7_permno, meta], ignore_index=True)
permno_list = mag7_permno['permno'].tolist()
mag7_permno

,permno,ticker,namedt,nameendt,comnam
0,14593,AAPL,1980-12-12,1982-10-31,APPLE COMPUTER INC
1,84788,AMZN,1997-05-15,2004-06-09,AMAZON COM INC
2,90319,GOOGL,2014-04-03,2015-04-23,GOOGLE INC
3,10107,MSFT,1986-03-13,2004-06-09,MICROSOFT CORP
4,86580,NVDA,1999-01-22,2004-06-09,NVIDIA CORP
5,93436,TSLA,2010-06-29,2017-02-01,TESLA MOTORS INC
6,13407,FB,2012-05-18,2019-09-16,FACEBOOK INC


In [4]:
permno_str = ",".join(map(str, permno_list))

query = f"""
SELECT DISTINCT gvkey, lpermno AS permno
FROM crsp.ccmxpf_linktable
WHERE lpermno IN ({permno_str})
  AND linktype IN ('LU','LC')
  AND linkprim IN ('P','C')
"""

link_df = db.raw_sql(query)
link_df['permno'] = link_df['permno'].astype(int)
gvkey_list = link_df['gvkey'].tolist()
gvkey_str = ",".join(f"'{g}'" for g in gvkey_list)

In [ ]:
query = f"""
SELECT permno,
    date,
    prc,
    ret,
    vol,
    shrout,
    cfacpr
FROM crsp.dsf
WHERE date >= '2010-01-01'
    AND date <= '2013-12-31'
    AND permno IN ({permno_str})
ORDER BY permno, date
"""

df_daily = db.raw_sql(query, date_cols=["date"])
df_daily["close_adj"] = df_daily["prc"].abs() / df_daily["cfacpr"]
df_daily = pd.merge(df_daily, mag7_permno[['permno', 'ticker']], on='permno', how='left')
df_daily.rename(columns={"ticker": "symbol", "vol": "volume"}, inplace=True)
df_daily['symbol'] = np.where(df_daily['symbol'] == 'FB', 'META', df_daily['symbol'])
prices = pd.concat(
    [prices, df_daily[['symbol', 'date', 'close_adj', 'volume']]]
).sort_values(["symbol", "date"]).reset_index(drop=True)
trade_dates = prices[['symbol', 'date']].drop_duplicates()

In [6]:
query = f"""
SELECT *
FROM comp.fundq
WHERE gvkey IN ({gvkey_str})
  AND datadate >= '2008-01-01'
  AND datadate <= '2025-12-31'
  AND indfmt = 'INDL'
  AND datafmt = 'STD'
  AND popsrc = 'D'
  AND consol = 'C'
ORDER BY datadate
"""

df = db.raw_sql(query, date_cols=["datadate"])

In [7]:
col = ['tic', 'rdq', 'datadate']
earnings = df.copy()
earnings = earnings[col + [c for c in earnings.columns if c not in col]]
earnings['rdq'] = pd.to_datetime(earnings['rdq'])
earnings.rename(columns={'rdq': 'date', 
                        'tic': 'symbol'}, inplace=True)
earnings['date'] = earnings.groupby(['symbol', 'fqtr'])['date'].ffill().bfill()
price_earnings = pd.merge(prices, earnings, on=['symbol', 'date'], how='outer')
price_earnings.sort_values(['symbol', 'date'], inplace=True)
price_earnings

,symbol,date,close_adj,volume,datadate,gvkey,fyearq,fqtr,fyr,indfmt,...,fic,hiid,cshtrq,dvpspq,dvpsxq,mkvaltq,prccq,prchq,prclq,adjex
0,AAPL,2008-04-23,<NA>,<NA>,2008-03-31,001690,2008,2,9,INDL,...,USA,01,2955007340.0,0.0,0.0,126485.3485,143.5,200.26,115.44,28.0
1,AAPL,2008-07-21,<NA>,<NA>,2008-06-30,001690,2008,3,9,INDL,...,USA,01,2167461160.0,0.0,0.0,148309.4777,167.44,192.24,143.61,28.0
2,AAPL,2008-10-21,<NA>,<NA>,2008-09-30,001690,2008,4,9,INDL,...,USA,01,2025043080.0,0.0,0.0,100967.1332,113.66,180.91,100.59,28.0
3,AAPL,2009-01-21,<NA>,<NA>,2008-12-31,001690,2009,1,9,INDL,...,USA,01,3034829654.0,0.0,0.0,75996.9203,85.35,116.4,79.14,28.0
4,AAPL,2009-04-22,<NA>,<NA>,2009-03-31,001690,2009,2,9,INDL,...,USA,01,1677798400.0,0.0,0.0,93757.7894,105.12,109.98,78.2,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27447,TSLA,2025-12-26,475.19,58780659.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
27448,TSLA,2025-12-29,459.64,66263033.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
27449,TSLA,2025-12-30,454.43,59238464.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
27450,TSLA,2025-12-31,449.72,49077961.0,NaT,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
cols_to_fill = price_earnings.columns.difference(['symbol', 'date'])
price_earnings = price_earnings.sort_values(['symbol', 'date'])

price_earnings[cols_to_fill] = (
    price_earnings
    .groupby('symbol')[cols_to_fill]
    .ffill()
)

price_earnings = pd.merge(trade_dates, price_earnings, on=['symbol', 'date'], how='left')
price_earnings = price_earnings.sort_values(['symbol', 'date']).reset_index(drop=True)
    
na_counts = price_earnings.groupby('symbol').apply(lambda x: x.isna().sum() / len(x))
maxna = na_counts.max()
price_earnings

,symbol,date,close_adj,volume,datadate,gvkey,fyearq,fqtr,fyr,indfmt,...,fic,hiid,cshtrq,dvpspq,dvpsxq,mkvaltq,prccq,prchq,prclq,adjex
0,AAPL,2010-01-04,7.643214,18241224.0,2009-09-30,001690,2009,4,9,INDL,...,USA,01,1035418930.0,0.0,0.0,166779.0421,185.35,188.9,134.42,28.0
1,AAPL,2010-01-05,7.656429,22082278.0,2009-09-30,001690,2009,4,9,INDL,...,USA,01,1035418930.0,0.0,0.0,166779.0421,185.35,188.9,134.42,28.0
2,AAPL,2010-01-06,7.534643,20396105.0,2009-09-30,001690,2009,4,9,INDL,...,USA,01,1035418930.0,0.0,0.0,166779.0421,185.35,188.9,134.42,28.0
3,AAPL,2010-01-07,7.520714,17628746.0,2009-09-30,001690,2009,4,9,INDL,...,USA,01,1035418930.0,0.0,0.0,166779.0421,185.35,188.9,134.42,28.0
4,AAPL,2010-01-08,7.570714,16553861.0,2009-09-30,001690,2009,4,9,INDL,...,USA,01,1035418930.0,0.0,0.0,166779.0421,185.35,188.9,134.42,28.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27386,TSLA,2025-12-24,485.4,41285429.0,2025-09-30,184996,2025,3,12,INDL,...,USA,01,5693189150.0,0.0,0.0,1478249.28,444.72,450.98,288.7701,1.0
27387,TSLA,2025-12-26,475.19,58780659.0,2025-09-30,184996,2025,3,12,INDL,...,USA,01,5693189150.0,0.0,0.0,1478249.28,444.72,450.98,288.7701,1.0
27388,TSLA,2025-12-29,459.64,66263033.0,2025-09-30,184996,2025,3,12,INDL,...,USA,01,5693189150.0,0.0,0.0,1478249.28,444.72,450.98,288.7701,1.0
27389,TSLA,2025-12-30,454.43,59238464.0,2025-09-30,184996,2025,3,12,INDL,...,USA,01,5693189150.0,0.0,0.0,1478249.28,444.72,450.98,288.7701,1.0


In [9]:
fig = px.line(
    price_earnings,
    x="date",
    y="close_adj",
    color="symbol",
    title="Stock Price Time Series"
)

fig.show()

In [10]:
col = ['symbol', 'date', 'close_adj', 'volume', 'datadate', 
'actq','ajexq','atq','capxy','ceqq','chechy','cheq','cogsq','cogsy',
'csh12q','cshfdq','cshiq','cshopq','cshoq','cshprq','cstkq','curcdq','datacqtr','datafqtr',
'dlcq','dltisy','dltry','dlttq','dpcy','dpq','dvpq','dvpspq','dvy',
'epsf12','epsfi12','epsfiq','epsfxq','epspi12','epspiq','epspxq','epsx12','fqtr','fyearq',
'fyr','ibadjq','ibcomq','ibcomy','ibcy','ibq','icaptq','invtq',
'lctq','ltq','mibq','miiq','miiy','niq','nopiq','oancfy','oepf12','oeps12',
'oepsxq','oepsxy','oiadpq','oiadpy','oibdpq','oibdpy','piq','ppentq',
'pstkq','pstkrq','rectq','req','revtq','revty','saleq','seqq',
'spiq','teqq','txdbq','txditcq','txpdy','txpq',
'txtq','xidocy','xidoy','xintq','xinty','xoprq','xrdq','xrdy','xsgaq','xsgay'
]
earnings_data = price_earnings[col].copy()
earnings_data

,symbol,date,close_adj,volume,datadate,actq,ajexq,atq,capxy,ceqq,...,txtq,xidocy,xidoy,xintq,xinty,xoprq,xrdq,xrdy,xsgaq,xsgay
0,AAPL,2010-01-04,7.643214,18241224.0,2009-09-30,31555.0,28.0,47501.0,1144.0,31640.0,...,1197.0,0.0,0.0,0.0,0.0,8320.0,358.0,1333.0,1421.0,5482.0
1,AAPL,2010-01-05,7.656429,22082278.0,2009-09-30,31555.0,28.0,47501.0,1144.0,31640.0,...,1197.0,0.0,0.0,0.0,0.0,8320.0,358.0,1333.0,1421.0,5482.0
2,AAPL,2010-01-06,7.534643,20396105.0,2009-09-30,31555.0,28.0,47501.0,1144.0,31640.0,...,1197.0,0.0,0.0,0.0,0.0,8320.0,358.0,1333.0,1421.0,5482.0
3,AAPL,2010-01-07,7.520714,17628746.0,2009-09-30,31555.0,28.0,47501.0,1144.0,31640.0,...,1197.0,0.0,0.0,0.0,0.0,8320.0,358.0,1333.0,1421.0,5482.0
4,AAPL,2010-01-08,7.570714,16553861.0,2009-09-30,31555.0,28.0,47501.0,1144.0,31640.0,...,1197.0,0.0,0.0,0.0,0.0,8320.0,358.0,1333.0,1421.0,5482.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27386,TSLA,2025-12-24,485.4,41285429.0,2025-09-30,64653.0,1.0,133735.0,6134.0,79970.0,...,570.0,0.0,0.0,76.0,253.0,24608.0,1630.0,4628.0,3192.0,8807.0
27387,TSLA,2025-12-26,475.19,58780659.0,2025-09-30,64653.0,1.0,133735.0,6134.0,79970.0,...,570.0,0.0,0.0,76.0,253.0,24608.0,1630.0,4628.0,3192.0,8807.0
27388,TSLA,2025-12-29,459.64,66263033.0,2025-09-30,64653.0,1.0,133735.0,6134.0,79970.0,...,570.0,0.0,0.0,76.0,253.0,24608.0,1630.0,4628.0,3192.0,8807.0
27389,TSLA,2025-12-30,454.43,59238464.0,2025-09-30,64653.0,1.0,133735.0,6134.0,79970.0,...,570.0,0.0,0.0,76.0,253.0,24608.0,1630.0,4628.0,3192.0,8807.0


In [11]:
fill_cols = ['cshopq', 'dlcq', 'dvpspq', 'txdbq', 'txditcq', 'xintq', 'xinty']
for col in fill_cols:
    earnings_data[col] = earnings_data[col].fillna(0)

In [12]:
(earnings_data.groupby('symbol').apply(lambda x: x.isna().sum() / len(x)))

,symbol,date,close_adj,volume,datadate,actq,ajexq,atq,capxy,ceqq,...,txtq,xidocy,xidoy,xintq,xinty,xoprq,xrdq,xrdy,xsgaq,xsgay
symbol,,,,,,,,,,,,,,,,,,,,,
AAPL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AMZN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GOOGL,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
META,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MSFT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NVDA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
TSLA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
earnings_data.to_pickle("mag7.pkl")